# Experimentation with preprocessing and GED rule based subsystem

- **Author**: Amir Anwar


## Preprocessing Interface

In [1]:
import json
import os
import sys

# NOTE: import from root of the project
sys.path.append(os.path.abspath("../../../../"))

from src.services.preprocessing.orchestrator import preprocess
from src.services.preprocessing.schemas import PreprocessingInput

In [2]:
pre_input = PreprocessingInput(text="ذهب أمير الى المنزل.")
pre_output = preprocess(pre_input)

[2026-06-14 18:32:27,801 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


In [3]:
print(json.dumps(pre_output.model_dump(), ensure_ascii=False, indent=2))

{
  "text": "ذهب أمير الى المنزل.",
  "normalized_text": "ذهب أمير الى المنزل.",
  "tokens": [
    {
      "index": 0,
      "form": "ذهب",
      "span": [
        0,
        3
      ],
      "norm_span": [
        0,
        3
      ],
      "affix_structure": "STEM"
    },
    {
      "index": 1,
      "form": "أمير",
      "span": [
        4,
        8
      ],
      "norm_span": [
        4,
        8
      ],
      "affix_structure": "STEM"
    },
    {
      "index": 2,
      "form": "الى",
      "span": [
        9,
        12
      ],
      "norm_span": [
        9,
        12
      ],
      "affix_structure": "STEM"
    },
    {
      "index": 3,
      "form": "المنزل",
      "span": [
        13,
        19
      ],
      "norm_span": [
        13,
        19
      ],
      "affix_structure": "DET+STEM"
    },
    {
      "index": 4,
      "form": ".",
      "span": [
        19,
        20
      ],
      "norm_span": [
        19,
        20
      ],
      "affix_structure"

---

## GED rule based testing with preprocessing


In [4]:
from src.services.ged.features.subsystems.rule_based import RuleBasedDetector
from src.services.ged.orchestrator import GEDService
from src.services.ged.schemas import GEDInput

2026-06-14 18:32:33.733 | DEBUG    | src.services.ged.features.subsystems.rule_based.registry:register_entry:99 - Registered entry: OT_ALIF_MAQSURA_PREP
2026-06-14 18:32:33.734 | DEBUG    | src.services.ged.features.subsystems.rule_based.registry:register_entry:99 - Registered entry: OT_TA_MARBUTA_NOUN
2026-06-14 18:32:33.735 | WARNING  | src.services.ged.features.subsystems.rule_based.loader:load_yaml_rules:193 - YAML file /home/amir/dev/baligh/src/services/ged/features/subsystems/rule_based/rules/punctuation.yaml does not contain a rule list , skipping.
2026-06-14 18:32:33.735 | INFO     | src.services.ged.features.subsystems.rule_based.loader:load_yaml_rules:230 - Loaded 2 YAML rules from /home/amir/dev/baligh/src/services/ged/features/subsystems/rule_based/rules.
2026-06-14 18:32:33.736 | DEBUG    | src.services.ged.features.subsystems.rule_based.detector:<module>:47 - Loaded 2 YAML rules from /home/amir/dev/baligh/src/services/ged/features/subsystems/rule_based/rules
2026-06-14 18

In [5]:
detector = RuleBasedDetector()
service = GEDService(subsystems=[detector])

In [6]:
def test_ged(text: str, sum_output: bool = True, show_preprocessing: bool = False):
    """Tests GED with preprocessing."""
    pre_input = PreprocessingInput(text=text)
    pre_output = preprocess(pre_input)

    ged_input = GEDInput(
        text=pre_output.text,
        normalized_text=pre_output.normalized_text,
        tokens=pre_output.tokens,
        morph_features=pre_output.morph_features,
    )
    ged_output = service.process(ged_input)

    if show_preprocessing:
        print("Preprocessing output:")
        print(json.dumps(pre_output.model_dump(), ensure_ascii=False, indent=2))

    if sum_output:
        if not ged_output.errors:
            print("No errors found")
        else:
            print(f"Found {len(ged_output.errors)} errors:")
            for i, error in enumerate(ged_output.errors, start=1):
                print("*" * 20 + f" Error {i} " + "*" * 20)
                print(f"  Error in: {ged_output.text[error.span[0] : error.span[1]]}")
                print(f"  Error type: {error.category}-{error.subtype}")
                print(f"  Error description: {error.explanation_text}")
    else:
        print(json.dumps(ged_output.model_dump(), ensure_ascii=False, indent=2))

In [7]:
test_ged("ذهب أمير الى المنزل.", sum_output=False)

{
  "text": "ذهب أمير الى المنزل.",
  "errors": [
    {
      "span": [
        9,
        12
      ],
      "token_refs": [
        2
      ],
      "category": "OT",
      "subtype": "hamza",
      "confidence": 1.0,
      "sources": [
        "rule_based"
      ],
      "provenance_tier": "tier_1_rule_derived",
      "explanation_eligible": true,
      "explanation_text": "حرف الجر أو الربط يبدأ بهمزة قطع (إ/أ) لا بألف مجردة (ا)؛ مثل: إلى، إن، أن , لا: الى، ان، ان"
    }
  ]
}


In [8]:
test_ged("ذهب أمير الى المنزل.")

Found 1 errors:
******************** Error 1 ********************
  Error in: الى
  Error type: OT-hamza
  Error description: حرف الجر أو الربط يبدأ بهمزة قطع (إ/أ) لا بألف مجردة (ا)؛ مثل: إلى، إن، أن , لا: الى، ان، ان


---

## Test Some rules


In [9]:
# OT_TA_MARBUTA_NOUN
test_ged("ذهب الطالب إلي الحديقه.")

Found 2 errors:
******************** Error 1 ********************
  Error in: إلي
  Error type: OT-alif_maqsura
  Error description: حرف الجر ينتهي بألف مقصورة (ى) لا ياء (ي)، مثل: على، إلى، حتى
******************** Error 2 ********************
  Error in: الحديقه
  Error type: OT-ta_marbuta
  Error description: الاسم المؤنث يُكتب بتاء مربوطة (ة) لا هاء (ه)


In [10]:
# OT_ALIF_MAQSURA_PREP
test_ged("ذهبت إلي المدرسة.")

Found 1 errors:
******************** Error 1 ********************
  Error in: إلي
  Error type: OT-alif_maqsura
  Error description: حرف الجر ينتهي بألف مقصورة (ى) لا ياء (ي)، مثل: على، إلى، حتى


In [11]:
# OT_HAMZA_PREP
test_ged("ذهبت الى المطار.")

Found 1 errors:
******************** Error 1 ********************
  Error in: الى
  Error type: OT-hamza
  Error description: حرف الجر أو الربط يبدأ بهمزة قطع (إ/أ) لا بألف مجردة (ا)؛ مثل: إلى، إن، أن , لا: الى، ان، ان


In [12]:
# PC_SPACE_BEFORE_PUNC : Multiple errors
test_ged("ذهبت إلى المطار ، وركبت الطائرة .")

Found 2 errors:
******************** Error 1 ********************
  Error in: ،
  Error type: PC-spacing
  Error description: علامة الترقيم يجب أن تلتصق بالكلمة التي تسبقها دون فراغ؛ مثل: «ذهب، ثم» لا «ذهب ، ثم»
******************** Error 2 ********************
  Error in: .
  Error type: PC-spacing
  Error description: علامة الترقيم يجب أن تلتصق بالكلمة التي تسبقها دون فراغ؛ مثل: «ذهب، ثم» لا «ذهب ، ثم»


In [13]:
# SY_VERB_SUBJECT_VSO
test_ged("ذهبوا المعلمون إلى المدرسة.")

Found 1 errors:
******************** Error 1 ********************
  Error in: ذهبوا
  Error type: SY-verb_subject_agreement
  Error description: إذا تقدَّم الفعل على الفاعل وجب إفراد الفعل وتجريده من علامة التثنية أو الجمع، مثل: «ذهب الطلاب» لا «ذهبوا الطلاب»


In [14]:
# SY_NOUN_ADJ_DEFINITENESS
test_ged("ذهبت إلى المدرسة جميلة الرائعة.")

Found 1 errors:
******************** Error 1 ********************
  Error in: الرائعة
  Error type: SY-noun_adjective_agreement
  Error description: النعت يتبع المنعوت في التعريف والتنكير؛ فإن كان الاسم معرفةً وجب تعريف النعت، وإن كان نكرةً وجب تنكيره


---
### Open testing

In [15]:
text = "ذهب أمير إلي المدرسة. الكتاب علي الطاولة. تعبت حتي تعلمت."
test_ged(text, sum_output=True, show_preprocessing=True)

Preprocessing output:
{
  "text": "ذهب أمير إلي المدرسة. الكتاب علي الطاولة. تعبت حتي تعلمت.",
  "normalized_text": "ذهب أمير إلي المدرسة. الكتاب علي الطاولة. تعبت حتي تعلمت.",
  "tokens": [
    {
      "index": 0,
      "form": "ذهب",
      "span": [
        0,
        3
      ],
      "norm_span": [
        0,
        3
      ],
      "affix_structure": "STEM"
    },
    {
      "index": 1,
      "form": "أمير",
      "span": [
        4,
        8
      ],
      "norm_span": [
        4,
        8
      ],
      "affix_structure": "STEM"
    },
    {
      "index": 2,
      "form": "إلي",
      "span": [
        9,
        12
      ],
      "norm_span": [
        9,
        12
      ],
      "affix_structure": "STEM"
    },
    {
      "index": 3,
      "form": "المدرسة",
      "span": [
        13,
        20
      ],
      "norm_span": [
        13,
        20
      ],
      "affix_structure": "DET+STEM"
    },
    {
      "index": 4,
      "form": ".",
      "span": [
        20,